## Trainning The Evaluation Function Weights


In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
import sys
print(sys.executable)

C:\Users\athar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe


### About Data
- data is taken from stockfish static evaluation (depth=0)
- our current evaluation function is static evaluation.
- we havent added check/checkmate evaluation so check positions are removed from dataset to reduce huge fluctuations in Error.

### Representing Data in 2D grid format

In [7]:

EMPTY = 0
# Piece Representations
BLACK_PAWN = -1
BLACK_ROOK = -2
BLACK_KNIGHT = -3
BLACK_BISHOP = -4
BLACK_QUEEN = -5
BLACK_KING = -6

WHITE_PAWN = 1
WHITE_ROOK = 2
WHITE_KNIGHT = 3
WHITE_BISHOP = 4
WHITE_QUEEN = 5
WHITE_KING = 6


def fromFEN(fen):
    # Creates an Empty Board of 8x8
    board = []
    for i in range(8):
        row=[]
        for j in range(8):
            row.append(EMPTY)
        board.append(row)

    row = 0
    col = 0

    for c in fen:
        # if c reaches an empty character then board representation ends
        if c==' ':
            break
        # if / is encountered move to next row
        if c == '/':
            row += 1
            col = 0
        # if a digit is encountered then skip that many consecutive squares
        elif c.isdigit():
            col += int(c)
        # if character is found then place it on current row and column
        else:
            if c == 'P':
                board[row][col] = WHITE_PAWN
            elif c == 'R':
                board[row][col] = WHITE_ROOK
            elif c == 'N':
                board[row][col] = WHITE_KNIGHT
            elif c == 'B':
                board[row][col] = WHITE_BISHOP
            elif c == 'Q':
                board[row][col] = WHITE_QUEEN
            elif c == 'K':
                board[row][col] = WHITE_KING

            elif c == 'p':
                board[row][col] = BLACK_PAWN
            elif c == 'r':
                board[row][col] = BLACK_ROOK
            elif c == 'n':
                board[row][col] = BLACK_KNIGHT
            elif c == 'b':
                board[row][col] = BLACK_BISHOP
            elif c == 'q':
                board[row][col] = BLACK_QUEEN
            elif c == 'k':
                board[row][col] = BLACK_KING

            col += 1

    return board


data=pd.read_csv("data_train.csv")
data_dev=pd.read_csv("data_dev.csv")
data_test=pd.read_csv("data_test.csv")
data=data.dropna()
data_dev=data_dev.dropna()
data_test=data_test.dropna()
tp=fromFEN(data["fen"][0])
tp

[[0, 0, 0, 2, 0, 0, 0, 0],
 [0, 0, 0, -6, 0, -1, 0, -1],
 [0, 0, 0, 0, 0, 0, -1, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 1, 1, 1],
 [0, 0, 0, 0, 0, 0, 6, 0]]

### getting features from current board representation

In [ ]:
import subprocess

X=[]

def getFeatures(dataFrame):
    boards = []

    for data_str in dataFrame["fen"]:
        arr = fromFEN(data_str)

        for row in arr:
            boards.append(" ".join(map(str, row)))

    inp = f"{len(dataFrame)}\n" + "\n".join(boards) + "\n"

    result = subprocess.run(
        ["../feature"],
        input=inp,
        text=True,
        capture_output=True
    )

    return [
        list(map(float, line.split()))
        for line in result.stdout.strip().splitlines()
    ]

X=getFeatures(data)

X_train=pd.DataFrame(
    X,
    columns=["material","mobility","pawns","pressure","threat","rook","check","KingSafety","kingPosition"]
)

X=getFeatures(data_dev)
X_dev=pd.DataFrame(
    X,
    columns=["material","mobility","pawns","pressure","threat","rook","check","KingSafety","kingPosition"]
)

X=getFeatures(data_test)
X_test=pd.DataFrame(
    X,
    columns=["material","mobility","pawns","pressure","threat","rook","check","KingSafety","kingPosition"]
)
X_train.head()

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [ ]:
X_test

,material,mobility,pawns,pressure,threat,rook,check,KingSafety,kingPosition
0,827.0,58.0,0.0,-10.0,0.0,0.0,0.0,-520.0,0.0
1,-500.0,-19.0,-15.0,0.0,0.0,0.0,0.0,-490.0,20.0
2,-140.0,8.0,-65.0,0.0,0.0,-15.0,0.0,-300.0,30.0
3,-87.0,23.0,-35.0,-10.0,0.0,0.0,0.0,-470.0,0.0
4,-165.0,-22.0,-35.0,60.0,80.0,-15.0,0.0,-590.0,0.0
...,...,...,...,...,...,...,...,...,...
4927,-125.0,30.0,0.0,-30.0,0.0,0.0,0.0,-370.0,0.0
4928,0.0,40.0,0.0,0.0,0.0,0.0,0.0,-20.0,0.0
4929,-433.0,18.0,0.0,0.0,-10.0,0.0,0.0,-950.0,0.0
4930,15.0,28.0,0.0,0.0,0.0,0.0,0.0,-600.0,0.0


In [5]:
scale=StandardScaler()
scale.fit(X_train)
np.save("scaler_mean.npy", scale.mean_)
np.save("scaler_scale.npy", scale.scale_)
X_train_scaled=scale.transform(X_train)
X_dev_scaled=scale.transform(X_dev)
X_test_scaled=scale.transform(X_test)


Y_train=data[["score"]]
Y_dev=data_dev[["score"]]
Y_test=data_test[["score"]]

model = Sequential([
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(64,activation='relu'),
    Dense(32,activation='relu'),
    Dense(20,activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)


model.fit(X_train_scaled,Y_train)
for i,layer in enumerate(model.layers):
    W,b=layer.get_weights()

    print("Layer",i)
    print("Weights:",W.shape)
    print("Bias:",b.shape)
Y_pred_dev=model.predict(X_dev_scaled)
Y_pred_test=model.predict(X_test_scaled)
Y_pred_train=model.predict(X_train_scaled)

err_train=root_mean_squared_error(Y_train,Y_pred_train)
err_test= root_mean_squared_error(Y_test,Y_pred_test)
err_dev = root_mean_squared_error(Y_dev,Y_pred_dev)


print("Dev set Error: ",err_dev)
print("Train set Error: ",err_train)
print("Test set Error:",err_test)

NameError: name 'X_train' is not defined

In [ ]:
import numpy as np

corr_dev = np.corrcoef(
    Y_dev.values.flatten(),
    Y_pred_dev.flatten()
)[0,1]

corr_train = np.corrcoef(
    Y_train.values.flatten(),
    Y_pred_train.flatten()
)[0,1]

corr_test = np.corrcoef(
    Y_test.values.flatten(),
    Y_pred_test.flatten()
)[0,1]

print("Correlation of Dev set:",corr_dev)
print("Correlation of Train set:",corr_train)
print("Correlation of Test set:",corr_test)

Correlation of Dev set: 0.8510631138480936
